In [1]:
# ============================================================
# CELLULE 1 — Test nba_api : connexion & endpoints disponibles
# ============================================================

from nba_api.stats.static import players, teams
from nba_api.live.nba.endpoints import scoreboard
import pandas as pd

# --- Équipes ---
toutes_equipes = teams.get_teams()
df_equipes = pd.DataFrame(toutes_equipes)
print(f"✅ Équipes NBA : {len(df_equipes)}")
print(df_equipes.columns.tolist())
print(df_equipes.head(3))

✅ Équipes NBA : 30
['id', 'full_name', 'abbreviation', 'nickname', 'city', 'state', 'year_founded']
           id            full_name abbreviation   nickname       city  \
0  1610612737        Atlanta Hawks          ATL      Hawks    Atlanta   
1  1610612738       Boston Celtics          BOS    Celtics     Boston   
2  1610612739  Cleveland Cavaliers          CLE  Cavaliers  Cleveland   

           state  year_founded  
0        Georgia          1949  
1  Massachusetts          1946  
2           Ohio          1970  


In [2]:
# ============================================================
# CELLULE 2 — Joueurs : volume, structure, champs disponibles
# ============================================================

tous_joueurs = players.get_players()
df_joueurs = pd.DataFrame(tous_joueurs)
print(f"✅ Joueurs NBA (base) : {len(df_joueurs)}")
print(df_joueurs.columns.tolist())
print(df_joueurs.head(3))

# Joueurs actifs uniquement
joueurs_actifs = [j for j in tous_joueurs if j['is_active']]
print(f"\n✅ Joueurs actifs : {len(joueurs_actifs)}")
print(pd.DataFrame(joueurs_actifs).head(3))

✅ Joueurs NBA (base) : 5024
['id', 'full_name', 'first_name', 'last_name', 'is_active']
      id            full_name first_name     last_name  is_active
0  76001       Alaa Abdelnaby       Alaa     Abdelnaby      False
1  76002      Zaid Abdul-Aziz       Zaid    Abdul-Aziz      False
2  76003  Kareem Abdul-Jabbar     Kareem  Abdul-Jabbar      False

✅ Joueurs actifs : 572
        id         full_name first_name last_name  is_active
0  1630173  Precious Achiuwa   Precious   Achiuwa       True
1   203500      Steven Adams     Steven     Adams       True
2  1628389       Bam Adebayo        Bam   Adebayo       True


In [3]:
# ============================================================
# CELLULE 3b — Diagnostic connexion directe
# ============================================================

import requests

# Test 1 : ping simple nba.com
try:
    r = requests.get("https://www.nba.com", timeout=10)
    print(f"✅ nba.com accessible : {r.status_code}")
except Exception as e:
    print(f"❌ nba.com inaccessible : {e}")

# Test 2 : endpoint stats direct
try:
    r = requests.get("https://stats.nba.com/stats/commonplayerinfo?PlayerID=2544", timeout=10)
    print(f"✅ stats.nba.com : {r.status_code}")
except Exception as e:
    print(f"❌ stats.nba.com inaccessible : {e}")

# Test 3 : balldontlie (alternative 100% gratuite, sans clé)
try:
    r = requests.get("https://api.balldontlie.io/v1/players?search=lebron", 
                     headers={"Authorization": ""},
                     timeout=10)
    print(f"✅ balldontlie : {r.status_code}")
except Exception as e:
    print(f"❌ balldontlie : {e}")

❌ nba.com inaccessible : HTTPSConnectionPool(host='www.nba.com', port=443): Read timed out. (read timeout=10)
❌ stats.nba.com inaccessible : HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=10)
✅ balldontlie : 401


In [5]:
# CELLULE 3c — Test cdn.nba.com (photos) + balldontlie avec clé
import requests

# Test photos joueurs
try:
    r = requests.get(
        "https://cdn.nba.com/headshots/nba/latest/1040x760/2544.png",
        timeout=10
    )
    print(f"✅ cdn.nba.com photos : {r.status_code} — taille : {len(r.content)} bytes")
except Exception as e:
    print(f"❌ cdn.nba.com : {e}")

# Test logos équipes
try:
    r = requests.get(
        "https://cdn.nba.com/logos/nba/1610612747/global/L/logo.svg",
        timeout=10
    )
    print(f"✅ cdn.nba.com logos : {r.status_code}")
except Exception as e:
    print(f"❌ cdn.nba.com logos : {e}")

✅ cdn.nba.com photos : 200 — taille : 214782 bytes
✅ cdn.nba.com logos : 200


In [7]:
# ============================================================
# CELLULE 4 — balldontlie : exploration complète
# ============================================================

import requests
import pandas as pd

CLE_API = "33b556a9-3f5b-4b4d-9244-29160c15f8e7"
BASE = "https://api.balldontlie.io/v1"
HEADERS = {"Authorization": CLE_API}

# --- 1. Joueurs ---
r = requests.get(f"{BASE}/players?search=lebron&per_page=5", headers=HEADERS, timeout=10)
data = r.json()
print("✅ Joueurs — champs disponibles :")
print(list(data['data'][0].keys()))
print(data['data'][0])

# --- 2. Équipes ---
r = requests.get(f"{BASE}/teams", headers=HEADERS, timeout=10)
data = r.json()
print("\n✅ Équipes — champs disponibles :")
print(list(data['data'][0].keys()))
print(data['data'][0])

# --- 3. Matchs (saison en cours) ---
r = requests.get(f"{BASE}/games?seasons[]=2024&per_page=3", headers=HEADERS, timeout=10)
data = r.json()
print("\n✅ Matchs — champs disponibles :")
print(list(data['data'][0].keys()))
print(data['data'][0])

# --- 4. Stats joueur ---
r = requests.get(f"{BASE}/stats?seasons[]=2024&player_ids[]=237&per_page=3", headers=HEADERS, timeout=10)
data = r.json()
print("\n✅ Stats — champs disponibles :")
if data['data']:
    print(list(data['data'][0].keys()))
    print(data['data'][0])

✅ Joueurs — champs disponibles :
['id', 'first_name', 'last_name', 'position', 'height', 'weight', 'jersey_number', 'college', 'country', 'draft_year', 'draft_round', 'draft_number', 'team']
{'id': 237, 'first_name': 'LeBron', 'last_name': 'James', 'position': 'F', 'height': '6-9', 'weight': '250', 'jersey_number': '23', 'college': 'St. Vincent-St. Mary HS (OH)', 'country': 'USA', 'draft_year': 2003, 'draft_round': 1, 'draft_number': 1, 'team': {'id': 14, 'conference': 'West', 'division': 'Pacific', 'city': 'Los Angeles', 'name': 'Lakers', 'full_name': 'Los Angeles Lakers', 'abbreviation': 'LAL'}}

✅ Équipes — champs disponibles :
['id', 'conference', 'division', 'city', 'name', 'full_name', 'abbreviation']
{'id': 1, 'conference': 'East', 'division': 'Southeast', 'city': 'Atlanta', 'name': 'Hawks', 'full_name': 'Atlanta Hawks', 'abbreviation': 'ATL'}

✅ Matchs — champs disponibles :
['id', 'date', 'season', 'status', 'period', 'time', 'postseason', 'postponed', 'home_team_score', 'visi

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [8]:
# CELLULE 4b — Diagnostic stats
r = requests.get(f"{BASE}/stats?seasons[]=2024&player_ids[]=237&per_page=3", headers=HEADERS, timeout=10)
print(f"Status : {r.status_code}")
print(f"Réponse brute : {r.text[:500]}")

Status : 401
Réponse brute : Unauthorized


In [9]:
# ============================================================
# CELLULE 6 — Test API alternative : sportsdata.io / ESPN API
# ============================================================

import requests

# ESPN API non officielle — gratuite, pas de clé
BASE_ESPN = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba"

# --- Matchs du jour ---
r = requests.get(f"{BASE_ESPN}/scoreboard", timeout=10)
print(f"ESPN scoreboard : {r.status_code}")
if r.status_code == 200:
    data = r.json()
    print(f"Clés disponibles : {list(data.keys())}")
    if 'events' in data:
        print(f"Matchs : {len(data['events'])}")
        if data['events']:
            print(list(data['events'][0].keys()))

# --- Infos équipe ---
r = requests.get(f"{BASE_ESPN}/teams/13", timeout=10)  # 13 = Lakers ESPN
print(f"\nESPN équipe : {r.status_code}")
if r.status_code == 200:
    data = r.json()
    print(list(data.keys()))

ESPN scoreboard : 200
Clés disponibles : ['leagues', 'season', 'day', 'events', 'provider']
Matchs : 1
['id', 'uid', 'date', 'name', 'shortName', 'season', 'competitions', 'links', 'status']

ESPN équipe : 200
['team']


In [10]:
# ============================================================
# CELLULE 7 — ESPN : exploration complète
# ============================================================

import requests, json

BASE = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba"
BASE2 = "https://sports.core.api.espn.com/v2/sports/basketball/leagues/nba"

# --- 1. Détail match du jour ---
r = requests.get(f"{BASE}/scoreboard", timeout=10)
data = r.json()
if data['events']:
    match = data['events'][0]
    compet = match['competitions'][0]
    print("✅ Détail match :")
    print(f"  Nom : {match['name']}")
    print(f"  Statut : {match['status']['type']['description']}")
    print(f"  Clés competition : {list(compet.keys())}")
    print(f"  Clés competitors : {list(compet['competitors'][0].keys())}")

# --- 2. Classement ---
r = requests.get(f"{BASE}/standings", timeout=10)
print(f"\n✅ Standings : {r.status_code}")
if r.status_code == 200:
    data = r.json()
    print(list(data.keys()))

# --- 3. Stats joueur via ESPN ---
r = requests.get(f"{BASE}/athletes/1966/stats", timeout=10)  # LeBron ESPN id
print(f"\n✅ Stats LeBron ESPN : {r.status_code}")
if r.status_code == 200:
    data = r.json()
    print(list(data.keys()))

# --- 4. Roster équipe (Lakers = 13) ---
r = requests.get(f"{BASE}/teams/13/roster", timeout=10)
print(f"\n✅ Roster Lakers : {r.status_code}")
if r.status_code == 200:
    data = r.json()
    print(list(data.keys()))
    if 'athletes' in data:
        print(f"  Nb joueurs : {len(data['athletes'])}")
        print(f"  Champs joueur : {list(data['athletes'][0].keys())}")

✅ Détail match :
  Nom : Oklahoma City Thunder at San Antonio Spurs
  Statut : Scheduled
  Clés competition : ['id', 'uid', 'date', 'attendance', 'type', 'timeValid', 'neutralSite', 'conferenceCompetition', 'playByPlayAvailable', 'recent', 'venue', 'competitors', 'notes', 'status', 'broadcasts', 'format', 'tickets', 'startDate', 'series', 'broadcast', 'geoBroadcasts', 'odds', 'highlights']
  Clés competitors : ['id', 'uid', 'type', 'order', 'homeAway', 'team', 'score', 'statistics', 'record', 'records', 'leaders']

✅ Standings : 200
['fullViewLink']

✅ Stats LeBron ESPN : 404

✅ Roster Lakers : 200
['timestamp', 'status', 'season', 'athletes', 'coach', 'team']
  Nb joueurs : 17
  Champs joueur : ['id', 'uid', 'guid', 'alternateIds', 'firstName', 'lastName', 'fullName', 'displayName', 'shortName', 'weight', 'displayWeight', 'height', 'displayHeight', 'age', 'dateOfBirth', 'debutYear', 'links', 'birthPlace', 'college', 'slug', 'headshot', 'jersey', 'position', 'injuries', 'teams', 'contr

In [11]:
# ============================================================
# CELLULE 8 — ESPN : stats joueurs + standings + photos
# ============================================================

import requests

BASE = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba"

# --- 1. Photo joueur via roster ---
r = requests.get(f"{BASE}/teams/13/roster", timeout=10)
data = r.json()
joueur = data['athletes'][0]
print("✅ Détail joueur :")
print(f"  Nom : {joueur['fullName']}")
print(f"  Photo : {joueur.get('headshot', {})}")
print(f"  Position : {joueur.get('position', {}).get('displayName')}")
print(f"  Blessure : {joueur.get('injuries')}")

# --- 2. Stats joueur — bon endpoint ---
espn_id = joueur['id']
r = requests.get(f"https://site.web.api.espn.com/apis/common/v3/sports/basketball/nba/athletes/{espn_id}/overview", timeout=10)
print(f"\n✅ Stats joueur ({joueur['fullName']}) : {r.status_code}")
if r.status_code == 200:
    data = r.json()
    print(list(data.keys()))

# --- 3. Standings correct ---
r = requests.get(
    "https://site.web.api.espn.com/apis/v2/sports/basketball/nba/standings",
    timeout=10
)
print(f"\n✅ Standings : {r.status_code}")
if r.status_code == 200:
    data = r.json()
    print(list(data.keys()))

# --- 4. Scoreboard historique (matchs passés) ---
r = requests.get(f"{BASE}/scoreboard?dates=20250601", timeout=10)
print(f"\n✅ Scoreboard date passée : {r.status_code}")
if r.status_code == 200:
    data = r.json()
    print(f"  Matchs : {len(data.get('events', []))}")

✅ Détail joueur :
  Nom : Deandre Ayton
  Photo : {'href': 'https://a.espncdn.com/i/headshots/nba/players/full/4278129.png', 'alt': 'Deandre Ayton'}
  Position : Center
  Blessure : []

✅ Stats joueur (Deandre Ayton) : 200
['statistics', 'news', 'nextGame', 'gameLog', 'rotowire', 'awards', 'fantasy']

✅ Standings : 200
['uid', 'id', 'name', 'abbreviation', 'shortName', 'children', 'isConference', 'season', 'links', 'seasons']

✅ Scoreboard date passée : 200
  Matchs : 0


In [12]:
# ============================================================
# CELLULE 9 — ESPN : stats détaillées + standings complets
# ============================================================

import requests

BASE_WEB = "https://site.web.api.espn.com/apis"

# --- 1. Stats détaillées joueur ---
r = requests.get(f"{BASE_WEB}/common/v3/sports/basketball/nba/athletes/4278129/overview", timeout=10)
data = r.json()

stats = data.get('statistics', {})
print("✅ Clés statistics :")
print(list(stats.keys()))
if 'splits' in stats:
    splits = stats['splits']
    print(f"  Clés splits : {list(splits.keys())}")
    if 'categories' in splits:
        for cat in splits['categories'][:2]:
            print(f"  Catégorie : {cat['name']}")
            print(f"  Stats : {[s['displayName'] for s in cat.get('stats', [])]}")

# --- 2. Standings complets ---
r = requests.get(f"{BASE_WEB}/v2/sports/basketball/nba/standings", timeout=10)
data = r.json()
conferences = data.get('children', [])
print(f"\n✅ Conférences : {len(conferences)}")
for conf in conferences:
    print(f"\n  {conf['name']} :")
    equipes = conf.get('standings', {}).get('entries', [])
    print(f"  Nb équipes : {len(equipes)}")
    if equipes:
        print(f"  Clés équipe : {list(equipes[0].keys())}")
        stats_equipe = equipes[0].get('stats', [])
        print(f"  Stats dispo : {[s['name'] for s in stats_equipe]}")

✅ Clés statistics :
['displayName', 'labels', 'names', 'displayNames', 'splits']


AttributeError: 'list' object has no attribute 'keys'

In [13]:
# ============================================================
# CELLULE 9b — ESPN : stats détaillées + standings complets
# ============================================================

import requests

BASE_WEB = "https://site.web.api.espn.com/apis"

# --- 1. Stats joueur ---
r = requests.get(f"{BASE_WEB}/common/v3/sports/basketball/nba/athletes/4278129/overview", timeout=10)
data = r.json()
stats = data.get('statistics', {})

print("✅ Labels stats disponibles :")
print(stats.get('displayNames', []))

splits = stats.get('splits', [])
print(f"\n✅ Nb splits : {len(splits)}")
if splits:
    print(f"  Clés premier split : {list(splits[0].keys())}")
    print(f"  Exemple : {splits[0]}")

# --- 2. Standings ---
r = requests.get(f"{BASE_WEB}/v2/sports/basketball/nba/standings", timeout=10)
data = r.json()
conferences = data.get('children', [])
print(f"\n✅ Conférences : {len(conferences)}")
for conf in conferences:
    print(f"\n  {conf['name']} :")
    equipes = conf.get('standings', {}).get('entries', [])
    print(f"  Nb équipes : {len(equipes)}")
    if equipes:
        print(f"  Clés : {list(equipes[0].keys())}")
        stats_eq = equipes[0].get('stats', [])
        print(f"  Stats dispo : {[s['name'] for s in stats_eq]}")
        break  # une conférence suffit pour le test

✅ Labels stats disponibles :
['Games Played', 'Minutes Per Game', 'Field Goal Percentage', '3-Point Field Goal Percentage', 'Free Throw Percentage', 'Rebounds Per Game', 'Assists Per Game', 'Blocks Per Game', 'Steals Per Game', 'Fouls Per Game', 'Turnovers Per Game', 'Points Per Game']

✅ Nb splits : 3
  Clés premier split : ['displayName', 'stats']
  Exemple : {'displayName': 'Regular Season', 'stats': ['72', '27.2', '67.1', '0.0', '64.5', '8.0', '0.8', '1.0', '0.6', '2.2', '1.2', '12.5']}

✅ Conférences : 2

  Eastern Conference :
  Nb équipes : 15
  Clés : ['team', 'stats']
  Stats dispo : ['avgPointsAgainst', 'avgPointsFor', 'clincher', 'differential', 'divisionWinPercent', 'gamesBehind', 'leagueWinPercent', 'losses', 'playoffSeed', 'pointDifferential', 'points', 'pointsAgainst', 'pointsFor', 'streak', 'winPercent', 'wins', 'gamesAhead', 'overall', 'Home', 'Road', 'vs. Div.', 'vs. Conf.', 'Last Ten Games']


In [15]:
# ============================================================
# CELLULE 10 — SYNTHÈSE : tableau récapitulatif des sources
# ============================================================

sources = {
    "Source": ["ESPN (non officielle)", "balldontlie (free)", "cdn.nba.com", "nba_api"],
    "Clé requise": ["Non", "Oui", "Non", "Non"],
    "Accessible FR": ["✅", "✅", "✅", "❌ Bloqué"],
    "Équipes": ["✅", "✅", "—", "✅"],
    "Joueurs": ["✅", "✅", "—", "✅"],
    "Photos joueurs": ["✅", "—", "✅ HD", "—"],
    "Logos équipes": ["—", "—", "✅ SVG", "—"],
    "Scores/Matchs": ["✅", "✅", "—", "—"],
    "Stats joueur": ["✅", "❌ Payant", "—", "❌ Bloqué"],
    "Standings": ["✅", "❌ Payant", "—", "—"],
    "Playoffs": ["✅", "✅", "—", "—"],
    "Blessures": ["✅", "—", "—", "—"],
    "Coût": ["0€", "0€ limité", "0€", "0€"],
}

import pandas as pd
df = pd.DataFrame(sources)
print(df.to_string(index=False))

print("\n🏆 STACK RETENUE :")
print("  • ESPN API non officielle → stats, scores, standings, roster, blessures")
print("  • cdn.nba.com → photos joueurs HD + logos équipes SVG")
print("  • balldontlie → backup scores/matchs si ESPN instable")
print("  • Coût total : 0€")

               Source Clé requise Accessible FR Équipes Joueurs Photos joueurs Logos équipes Scores/Matchs Stats joueur Standings Playoffs Blessures      Coût
ESPN (non officielle)         Non             ✅       ✅       ✅              ✅             —             ✅            ✅         ✅        ✅         ✅        0€
   balldontlie (free)         Oui             ✅       ✅       ✅              —             —             ✅     ❌ Payant  ❌ Payant        ✅         — 0€ limité
          cdn.nba.com         Non             ✅       —       —           ✅ HD         ✅ SVG             —            —         —        —         —        0€
              nba_api         Non      ❌ Bloqué       ✅       ✅              —             —             —     ❌ Bloqué         —        —         —        0€

🏆 STACK RETENUE :
  • ESPN API non officielle → stats, scores, standings, roster, blessures
  • cdn.nba.com → photos joueurs HD + logos équipes SVG
  • balldontlie → backup scores/matchs si ESPN instable


In [16]:
# CELLULE 10b — Tableau corrigé
sources = {
    "Source": ["ESPN (non officielle)", "balldontlie (free)", "cdn.nba.com"],
    "Clé requise": ["Non", "Oui", "Non"],
    "Équipes": ["✅", "✅", "—"],
    "Joueurs": ["✅", "✅", "—"],
    "Photos joueurs": ["✅ ESPN CDN", "—", "✅ NBA CDN"],
    "Logos équipes": ["✅", "—", "✅ SVG"],
    "Scores/Matchs": ["✅", "✅ backup", "—"],
    "Stats joueur": ["✅", "❌", "—"],
    "Standings": ["✅", "❌", "—"],
    "Playoffs": ["✅", "✅", "—"],
    "Blessures": ["✅", "—", "—"],
    "Coût": ["0€", "0€", "0€"],
}

import pandas as pd
print(pd.DataFrame(sources).to_string(index=False))

print("\n🏆 STACK RETENUE :")
print("  • ESPN API → tout : stats, scores, standings, roster, blessures, photos")
print("  • cdn.nba.com → photos HD + logos SVG en backup/complément")
print("  • balldontlie → backup scores uniquement")
print("  • Coût total : 0€")

               Source Clé requise Équipes Joueurs Photos joueurs Logos équipes Scores/Matchs Stats joueur Standings Playoffs Blessures Coût
ESPN (non officielle)         Non       ✅       ✅     ✅ ESPN CDN             ✅             ✅            ✅         ✅        ✅         ✅   0€
   balldontlie (free)         Oui       ✅       ✅              —             —      ✅ backup            ❌         ❌        ✅         —   0€
          cdn.nba.com         Non       —       —      ✅ NBA CDN         ✅ SVG             —            —         —        —         —   0€

🏆 STACK RETENUE :
  • ESPN API → tout : stats, scores, standings, roster, blessures, photos
  • cdn.nba.com → photos HD + logos SVG en backup/complément
  • balldontlie → backup scores uniquement
  • Coût total : 0€
